# City of Boston Property Assessment Data

The goal of this poject is to answer the following questions:
- Does owner occupancy decline after the '08 financial crash and if so, was it everywhere in Boston or only certain neighborhoods/zip codes? 
- Also, was this accelerating an already occurring trend or reversing a more equitable division of wealth? 
- Also also regarding owners: what is the breakdown between owned by a person, owned by a trust, and owned by a company (and then further breaking down the kinds of companies)?

Questions after seeing the data:
- What is the worth of a square ft in somewhere like Beacon Hill versus somewhere like Mattapan?
- What is the ratio of owner occupied units in a places like Beacon Hill versus places like Mattapan?
- How many of out-of-state landlords are there?
- Adding in utility data, can we see which housing units are not occupied?

Data Source: https://data.boston.gov/dataset/property-assessment

Issues to resolve:
    drop unneeded cols
    add column for FY
    rename columns
    fix known issues:
        convert zip code to int
        add back the dropped leading 0 to the zips
        move the decimal in gross_tax two places to the left
        do lot size/land sf and gross area need decimal moved?  
        separate mailing city state column into mailing city and mailing state columns
    concat all dfs together

In [16]:
import pandas as pd
import datetime as dt
import os
from sqlalchemy import create_engine

### Figure out which columns need to be dropped

In [17]:
# grab all file names in directory
cobDirectory = os.listdir()
cobCSVDirectory = []

#pare the list down to only the property assessment files
for item in cobDirectory:
    if '.csv' in item:
        cobCSVDirectory.append(item)
        
fullColNameList = []

#create a list of all unique column names, sorted alphabetically
for e in cobCSVDirectory:
    colNameList = list(pd.read_csv(e, nrows=1))
    for element in colNameList:
        fullColNameList.append(element)
    
fullColNameList = list(set(fullColNameList))

fullColNameList.sort()

print(fullColNameList)

[' GROSS_TAX ', 'AC_TYPE', 'AV_BLDG', 'AV_LAND', 'AV_TOTAL', 'BDRM_COND', 'BED_RMS', 'BLDG_AREA', 'BLDG_SEQ', 'BLDG_TYPE', 'BLDG_VALUE', 'BTHRM_STYLE1', 'BTHRM_STYLE2', 'BTHRM_STYLE3', 'CD_FLOOR', 'CITY', 'CM_ID', 'COM_UNITS', 'CORNER_UNIT', 'EXT_COND', 'EXT_FINISHED', 'EXT_FNISHED', 'FIREPLACES', 'FIRE_PLACE', 'FULL_BTH', 'FY2004_BLDG', 'FY2004_TOTAL', 'FY2006_BLDG', 'FY2006_LAND', 'FY2006_TOTAL', 'FY2007_BLDG', 'FY2007_LAND', 'FY2007_TOTAL', 'FY2008_BLDG', 'FY2008_GROSS_TAX', 'FY2008_LAND', 'FY2008_TOTAL', 'FY200_ LAND', 'GIS_ID', 'GROSS_AREA', 'GROSS_TAX', 'HEAT_FUEL', 'HEAT_SYSTEM', 'HEAT_TYPE', 'HLF_BTH', 'INT_COND', 'INT_WALL', 'KITCHEN', 'KITCHENS', 'KITCHEN_STYLE1', 'KITCHEN_STYLE2', 'KITCHEN_STYLE3', 'KITCHEN_TYPE', 'LAND_SF', 'LAND_VALUE', 'LATITUDE', 'LIVING _AREA', 'LIVING_AREA', 'LONGITUDE', 'LOTSIZE', 'LU', 'LUC', 'LU_DESC', 'Location', 'MAIL CS', 'MAIL_ADDRESS', 'MAIL_ADDRESSEE', 'MAIL_CITY', 'MAIL_CITY_STATE', 'MAIL_CS', 'MAIL_STATE', 'MAIL_STREET_ADDRESS', 'MAIL_ZIP', 

In [18]:
#compared column names with property assessment data key PDFs to see which columns need to be kept and which to group together
# 'LUC' stands for Land Use Code and the code descriptions don't 100% align with the LU_DESC values but the LU_DESC values based on spot checking are more detailed
# 'SFYI_VALUE' is not listed as a data key anywhere, but may stand for Special Features Yard Items Value
# GROSS_TAX col note for some years: Amount is based upon the total assessed value multiplied by the tax rate for the given year. The gross tax amount does not include any personal exemptions.  The value is stored in a non-decimal format.  Hence, $1,654.23 is displayed as 165423 .
colsToBeKept = ['OWNER_MAIL_CS', 'Owner_MAIL_CS', 'MAIL_CITY_STATE',  'MAIL_CS', 'MAIL CS', 'BLDG_TYPE', 'Fiscal_Year',
                'PID',  'Parcel_ID', 'CM_ID', 'GIS_ID', 'UNIT_NUM', 'CITY', 'LU', 'LU_DESC', 'OWN_OCC',
               'ST_NUM_CHAR', 'ST_NUM', 'ST_NAME', 'ST_NAME_SFX', 'ST_NAME_SUF', 'ST_SFX', 'ZIPCODE','ZIP_CODE',
               'OWNER', 'OWNER FY04', 'OWNER FY07', 'MAIL_ADDRESSEE', 'MAIL_CITY', 'MAIL_STATE', 'LOTSIZE','LAND_SF',
               'MAIL_STREET_ADDRESS','Owner_MAIL_ADDRESS', 'OWNER MAIL ADDRESS', 'OWNER_MAIL_ADDRESS', 'MAIL_ADDRESS',
               'MAIL_ZIP', 'MAIL_ZIPCODE', 'MAIL_ZIP_CODE',  'OWNER_MAIL_ZIPCODE', 'Owner_MAIL_ZIPCODE',
               'BLDG_AREA', 'GROSS_AREA', 'LIVING _AREA', 'LIVING_AREA', ' GROSS_TAX ', 'GROSS_TAX', 'FY2008_GROSS_TAX',
               'LAND_VALUE', 'FY200_ LAND', 'FY2006_LAND', 'FY2007_LAND', 'FY2008_LAND', 'AV_LAND',
               'BLDG_VALUE', 'FY2004_BLDG', 'FY2006_BLDG', 'FY2007_BLDG', 'FY2008_BLDG', 'AV_BLDG',
               'TOTAL_VALUE', 'FY2004_TOTAL', 'FY2006_TOTAL', 'FY2007_TOTAL', 'FY2008_TOTAL', 'AV_TOTAL']

### Figure out which columns need to be renamed

In [19]:
#compared column names with property assessment data key PDFs to see which columns need to be grouped together
ownerMailingAddressCityState = ['OWNER_MAIL_CS', 'Owner_MAIL_CS', 'MAIL_CITY_STATE',  'MAIL_CS', 'MAIL CS']

renameCols = {'ParcelID': ['PID',  'Parcel_ID'],
 'CondoMainID': 'CM_ID',
 'GeographicInformationSystemID': 'GIS_ID',
 'StreetNumber': ['ST_NUM_CHAR', 'ST_NUM'],
 'StreetNumber2': 'ST_NUM2',
 'StreetName': 'ST_NAME',
 'StreetSuffix': ['ST_NAME_SFX', 'ST_NAME_SUF', 'ST_SFX'], 
 'UnitNumber': 'UNIT_NUM',
 'City': 'CITY',
 'ZipCode': ['ZIPCODE','ZIP_CODE'],
 'LandUse' : 'LU',
 'LandUseDescription': 'LU_DESC',
 'BuildingType': 'BLDG_TYPE',
 'OwnerOccupied': 'OWN_OCC',
 'Owner': ['OWNER', 'OWNER FY04', 'OWNER FY07'],
 'OwnerMailingAddressName': 'MAIL_ADDRESSEE',
 'OwnerMailingAddressStreet': ['MAIL_STREET_ADDRESS','Owner_MAIL_ADDRESS', 'OWNER MAIL ADDRESS', 'OWNER_MAIL_ADDRESS', 'MAIL_ADDRESS'],
 'OwnerMailingAddressCity': 'MAIL_CITY',
 'OwnerMailingAddressState': 'MAIL_STATE',
 'OwnerMailingAddressZipCode': ['MAIL_ZIP', 'MAIL_ZIPCODE', 'MAIL_ZIP_CODE',  'OWNER_MAIL_ZIPCODE', 'Owner_MAIL_ZIPCODE'],
 'LandSquareFoot': ['LOTSIZE','LAND_SF'],
 'BuildingArea':'BLDG_AREA', 
 'GrossArea': 'GROSS_AREA',
 'LivingArea': ['LIVING _AREA', 'LIVING_AREA'],
 'LandValue': ['LAND_VALUE', 'FY200_ LAND', 'FY2006_LAND', 'FY2007_LAND', 'FY2008_LAND', 'AV_LAND'],
 'BuildingValue': ['BLDG_VALUE', 'FY2004_BLDG', 'FY2006_BLDG', 'FY2007_BLDG', 'FY2008_BLDG', 'AV_BLDG'],
 'TotalValue': ['TOTAL_VALUE', 'FY2004_TOTAL', 'FY2006_TOTAL', 'FY2007_TOTAL', 'FY2008_TOTAL', 'AV_TOTAL'],
 'GrossTax': [' GROSS_TAX ', 'GROSS_TAX', 'FY2008_GROSS_TAX']}

In [20]:
def readInData(fileName, year):
    #read in data
    df = pd.read_csv(fileName)
    #add in fiscal year column
    df['Fiscal_Year'] = year
    #remove unneeded columns
    df = df[df.columns.intersection(colsToBeKept)]
    return df

In [21]:
pa2004 = readInData('data2004-lite.csv', 2004)
pa2005 = readInData('data2005-lite.csv', 2005)
pa2006 = readInData('data2006lite.csv', 2006)
pa2007 = readInData('fy2007.csv', 2007)
pa2008 = readInData('property-assessment-fy08.csv', 2008)
pa2009 = readInData('property-assessment-fy09.csv', 2009)
pa2010 = readInData('property-assessment-fy10.csv', 2010)
pa2011 = readInData('property-assessment-fy11.csv', 2011)
pa2012 = readInData('property-assessment-fy12.csv', 2012)
pa2013 = readInData('property-assessment-fy13.csv', 2013)
pa2014 = readInData('property-assessment-fy2014.csv', 2014)
pa2015 = readInData('property-assessment-fy2015.csv', 2015)
pa2016 = readInData('property-assessment-fy2016.csv', 2016)
pa2017 = readInData('property-assessment-fy2017.csv', 2017)
pa2018 = readInData('ast2018full.csv', 2018)
pa2019 = readInData('fy19fullpropassess.csv', 2019)
pa2020 = readInData('data2020-full.csv', 2020)
pa2021 = readInData('data2021-full.csv', 2021)
pa2022 = readInData('fy2022pa-4.csv', 2022)
pa2023 = readInData('fy2023-property-assessment-data.csv', 2023)
pa2024 = readInData('fy2024-property-assessment-data_1_5_2024.csv', 2024)
pa2025 = readInData('fy2025-property-assessment-data_12_30_2024.csv', 2025)
pa2026 = readInData('fy2026-property-assessment-data_rev.csv', 2026)

/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (13) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (6,13) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (13,25,26,27,33,34,37,41,44,45,48,49,50,51,52) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/Users/jbryant/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3263: DtypeWarning: Columns (13,42,46,49) have mixed types.Specify dtype option on import or set low_memory=False

In [22]:
def renameColumns(df):
    for col in df:
        for key, value in renameCols.items():
            if col in value:
                df.rename(columns={col: key}, inplace=True)
    return df

In [23]:
pa2004 = renameColumns(pa2004)
pa2005 = renameColumns(pa2005)
pa2006 = renameColumns(pa2006)
pa2007 = renameColumns(pa2007)
pa2008 = renameColumns(pa2008)
pa2009 = renameColumns(pa2009)
pa2010 = renameColumns(pa2010)
pa2011 = renameColumns(pa2011)
pa2012 = renameColumns(pa2012)
pa2013 = renameColumns(pa2013)
pa2014 = renameColumns(pa2014)
pa2015 = renameColumns(pa2015)
pa2016 = renameColumns(pa2016)
pa2017 = renameColumns(pa2017)
pa2018 = renameColumns(pa2018)
pa2019 = renameColumns(pa2019)
pa2020 = renameColumns(pa2020)
pa2021 = renameColumns(pa2021)
pa2022 = renameColumns(pa2022)
pa2023 = renameColumns(pa2023)
pa2024 = renameColumns(pa2024)
pa2025 = renameColumns(pa2025)
pa2026 = renameColumns(pa2026)